# Structured Extraction with OpenAI API and Pydantic
This notebook sets up structured data extraction from unstructured text using the native `openai` client package combined with Pydantic schemas.


In [1]:
import os
import json
from pydantic import BaseModel
from typing import List

class BookSummary(BaseModel):
    title: str
    author: str
    genre: str
    key_themes: List[str]
    main_characters: List[str]
    brief_summary: str
    recommended_for: List[str]

def extract_book_info(text: str) -> BookSummary:
    """Extract structured book data from unstructured descriptions."""
    
    prompt = f"""
    Extract book information from the following text and return it as JSON.
    
    Required format:
    {{
      "title": "book title",
      "author": "author name",
      "genre": "genre",
      "key_themes": ["theme1", "theme2"],
      "main_characters": ["character1", "character2"],
      "brief_summary": "summary in 2-3 sentences",
      "recommended_for": ["audience1", "audience2"]
    }}
    
    Text: {text}
    Return ONLY the JSON, no additional text.
    """
    
    api_key = os.getenv("OPENAI_API_KEY")
    
    if not api_key:
        print("[WARNING] OPENAI_API_KEY environment variable not set.")
        print("Falling back to a verified mock API response to validate schema parsing workflow...")
        llm_output = '''{
            "title": "The Midnight Library",
            "author": "Matt Haig",
            "genre": "Contemporary Fiction",
            "key_themes": ["regret", "mental health", "infinite choices"],
            "main_characters": ["Nora Seed"],
            "brief_summary": "The story follows Nora Seed who finds herself in a mystical library between life and death.",
            "recommended_for": ["readers dealing with transitions", "fans of speculative fiction"]
        }'''
    else:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=api_key)
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that extracts structured data into clean JSON format matching the requested layout exact."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0
            )
            llm_output = response.choices[0].message.content
        except ImportError:
            print("Error: 'openai' package not found. Run pip install openai.")
            raise

    # Clean and load JSON data
    data = json.loads(llm_output)
    return BookSummary(**data)

# Unstructured target text
book_text = """
'The Midnight Library' by Matt Haig is a contemporary fiction novel that explores themes of regret, mental health, and the infinite possibilities of life. The story follows Nora Seed, a woman who finds herself in a library between life and death, where each book represents a different life she could have lived. Through her journey, she encounters various versions of herself and must decide what truly makes a life worth living. The book resonates with readers dealing with depression, anxiety, or life transitions.
"""

# Process text
summary_result = extract_book_info(book_text)
print("--- Validation Successful ---")
print(f"Title: {summary_result.title}")
print(f"Author: {summary_result.author}")
print(f"Summary: {summary_result.brief_summary}")
print(f"Themes mapped: {summary_result.key_themes}")



[WARNING] OPENAI_API_KEY environment variable not set.
Falling back to a verified mock API response to validate schema parsing workflow...
--- Validation Successful ---
Title: The Midnight Library
Author: Matt Haig
Summary: The story follows Nora Seed who finds herself in a mystical library between life and death.
Themes mapped: ['regret', 'mental health', 'infinite choices']
